In [ ]:
import json
import pandas as pd
import numpy as np

from pathlib import Path

In [ ]:
data_folder = Path("data_5000")

json_files = sorted(data_folder.glob("*.json"))

print("Total game files:", len(json_files))
assert len(json_files)==5000, "Expected exactly 5,000 new games in data_5000/"

In [ ]:
with open(json_files[0], "r") as f:
    game = json.load(f)

print(game.keys())

In [ ]:
for record in game["action_records"][:16]:
    print(record[0])

In [ ]:
def extract_initial_placements(game):

    game_id = game["game"]["id"]
    colors = game["colors"]

    players = {}

    # Initialize player records
    for order, color in enumerate(colors, start=1):

        players[color] = {
            "Game_ID": game_id,
            "Player": color,
            "Placement_Order": order,
            "Settlement_1": None,
            "Road_1": None,
            "Settlement_2": None,
            "Road_2": None
        }

    # Only extract the first 16 actions
    actions = [
        record[0]
        for record in game["action_records"][:16]
    ]

    for color, action_type, value in actions:

        player = players[color]

        if action_type == "BUILD_SETTLEMENT":

            if player["Settlement_1"] is None:
                player["Settlement_1"] = value

            else:
                player["Settlement_2"] = value

        elif action_type == "BUILD_ROAD":

            if player["Road_1"] is None:
                player["Road_1"] = value

            else:
                player["Road_2"] = value

        else:
            raise ValueError(
                f"Unexpected setup action: {action_type}"
            )

    # Return four player records
    return list(players.values())

In [ ]:
records = extract_initial_placements(game)

df_test = pd.DataFrame(records)

display(df_test)

In [ ]:
all_records = []

for file in json_files:

    with open(file, "r") as f:
        game = json.load(f)

    records = extract_initial_placements(game)

    all_records.extend(records)

df = pd.DataFrame(all_records)

display(df.head(10))

In [ ]:
print("Total games:", df["Game_ID"].nunique())

print("Total records:", len(df))

print("Dataset shape:", df.shape)

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated(
    subset=["Game_ID", "Player"]
).sum()

In [ ]:
df.groupby("Game_ID").size().value_counts()

In [ ]:
for i in [1, 2]:

    valid = df.apply(
        lambda row: row[f"Settlement_{i}"]
        in row[f"Road_{i}"],
        axis=1
    )

    print(f"Round {i} invalid roads:", (~valid).sum())

In [ ]:
map_records = []

for file in json_files:

    with open(file, "r") as f:
        game = json.load(f)

    game_id = game["game"]["id"]

    for tile in game["map"]["tiles"]:

        if tile["type"] == "LAND":

            map_records.append({
                "Game_ID": game_id,
                "Tile_ID": tile["id"],
                "Coordinate_X": tile["coordinate"][0],
                "Coordinate_Y": tile["coordinate"][1],
                "Coordinate_Z": tile["coordinate"][2],
                "Resource": tile["resource"],
                "Number": tile["number"]
            })

df_map = pd.DataFrame(map_records)

display(df_map.head(10))

In [ ]:
output_folder = Path("processed_data_5000")

output_folder.mkdir(exist_ok=True)

df.to_csv(
    output_folder / "initial_placements.csv",
    index=False
)

df_map.to_csv(
    output_folder / "map_tiles.csv",
    index=False
)